# PHẦN III — TẠO CONTROLLER VM TEMPLATE

## Mục tiêu phần này

Phần này tạo một **template image cho Controller VM** dựa trên Ubuntu 24.04 Cloud Image.

Template này sẽ được dùng để clone ra VM Controller phục vụ triển khai OpenStack bằng Kolla-Ansible.

## Kết quả mong muốn sau phần III

Sau khi hoàn thành, ta có file:

```bash
/var/lib/libvirt/templates/tpl-controller.img
```

Template này đã có sẵn:

- Ubuntu 24.04 Cloud Image
- Disk virtual size 100GB
- SSH server + cloud-init
- Docker
- Python venv `/opt/kolla-venv`
- Kolla-Ansible
- Chrony
- User `ubuntu` có quyền sudo không cần password
- SSH password login được bật
- SSH public key đã được inject
- Template đã được sysprep để clone VM sạch

## Lưu ý quan trọng

- Thực hiện trên host vật lý đã cài `libguestfs-tools`.
- Không chạy trực tiếp template như VM thường xuyên; template chỉ dùng để clone.
- Sau khi sysprep, image nên để read-only bằng `chmod 444`.
- Nếu `virt-customize` lỗi permission, kiểm tra KVM/libvirt và quyền truy cập file image.


## BƯỚC 12 — Tải Ubuntu 24.04 Cloud Image

Ubuntu Cloud Image là base image dạng QCOW2, rất phù hợp để tạo VM bằng cloud-init.

Ta sẽ tải image vào thư mục:

```bash
/var/lib/libvirt/templates
```

File base image:

```bash
ubuntu-24.04-base.img
```


In [ ]:
# ============================================================
# BƯỚC 12 — Tải Ubuntu 24.04 Cloud Image
# Chạy trên host vật lý / server dùng để tạo VM template
# ============================================================

# Tạo thư mục chứa template image
sudo mkdir -p /var/lib/libvirt/templates

# Gán quyền cho user hiện tại để thao tác thuận tiện
sudo chown -R $USER:$USER /var/lib/libvirt/templates

# Di chuyển vào thư mục template
cd /var/lib/libvirt/templates

# Tải Ubuntu 24.04 Noble Cloud Image
# -c: tiếp tục download nếu bị ngắt
# -O: lưu với tên chuẩn ubuntu-24.04-base.img
wget -c https://cloud-images.ubuntu.com/noble/current/noble-server-cloudimg-amd64.img \
  -O ubuntu-24.04-base.img

# Tải file SHA256SUMS để kiểm tra checksum
wget -c https://cloud-images.ubuntu.com/noble/current/SHA256SUMS

# Xác minh checksum image vừa tải
# Kết quả mong muốn:
# ubuntu-24.04-base.img: OK
sha256sum -c SHA256SUMS --ignore-missing

# Xem thông tin image
# Kết quả mong muốn:
# file format: qcow2
# virtual size: khoảng vài GiB ban đầu
qemu-img info ubuntu-24.04-base.img


### Verify Bước 12

Chạy các lệnh sau để xác nhận file base image tồn tại và đúng định dạng.


In [ ]:
# Verify file đã tồn tại
ls -lh /var/lib/libvirt/templates/ubuntu-24.04-base.img

# Verify định dạng image
qemu-img info /var/lib/libvirt/templates/ubuntu-24.04-base.img | egrep "file format|virtual size|disk size"

# Nếu checksum không OK:
# 1. Xóa image cũ
# 2. Tải lại bằng wget -c
# 3. Chạy lại sha256sum


## BƯỚC 13 — Tạo và resize Controller Template

Ở bước này ta copy base image thành template riêng cho Controller.

Không sửa trực tiếp file base image để sau này còn dùng lại cho Compute hoặc các VM khác.

File tạo ra:

```bash
tpl-controller.img
```

Dung lượng virtual resize lên:

```bash
100G
```

Lưu ý: `qemu-img resize` chỉ tăng virtual size. Dung lượng thật trên disk vẫn tăng dần theo dữ liệu ghi vào image.


In [ ]:
# ============================================================
# BƯỚC 13 — Tạo template controller và resize lên 100GB
# ============================================================

cd /var/lib/libvirt/templates

# Copy base image sang template controller
cp ubuntu-24.04-base.img tpl-controller.img

# Resize virtual disk lên 100GB
# Đây là virtual size, không có nghĩa là file chiếm ngay 100GB vật lý
qemu-img resize tpl-controller.img 100G

# Kiểm tra virtual size
# Kết quả mong muốn:
# virtual size: 100 GiB
qemu-img info tpl-controller.img | grep "virtual size"


### Verify Bước 13

Xác nhận image `tpl-controller.img` đã được tạo và có virtual size 100GB.


In [ ]:
# Verify template tồn tại
ls -lh /var/lib/libvirt/templates/tpl-controller.img

# Verify virtual size
qemu-img info /var/lib/libvirt/templates/tpl-controller.img

# Nếu resize sai:
# Có thể xóa tpl-controller.img và copy lại từ ubuntu-24.04-base.img
# rm -f tpl-controller.img
# cp ubuntu-24.04-base.img tpl-controller.img
# qemu-img resize tpl-controller.img 100G


## BƯỚC 14 — `virt-customize`: cài package vào Controller Template

`virt-customize` cho phép chỉnh sửa image offline, chưa cần boot VM.

Bước này sẽ cài sẵn:

- Tool hệ thống: `curl`, `wget`, `git`, `vim`, `htop`, `net-tools`
- Python toolchain: `python3-pip`, `python3-venv`, `python3-dev`
- SSH server
- Cloud-init
- Chrony
- Docker
- MariaDB client
- Kolla-Ansible trong `/opt/kolla-venv`

## Lưu ý quan trọng

- Bước này có thể mất 10–20 phút tùy tốc độ mạng.
- Cần Internet từ host vì image sẽ update/install package.
- Nếu host dùng proxy hoặc mạng lab bị chặn, bước này có thể fail khi apt/pip.
- Nếu muốn bản ổn định hơn, có thể pin version Kolla-Ansible theo release đang dùng.


In [ ]:
# ============================================================
# BƯỚC 14 — virt-customize cài package vào tpl-controller.img
# ============================================================

cd /var/lib/libvirt/templates

# Cài package nền tảng, Docker, Kolla-Ansible, Chrony vào image
sudo virt-customize -a tpl-controller.img \
  --update \
  --install curl,wget,git,vim,htop,net-tools,python3-pip,python3-venv,python3-dev,openssh-server,cloud-init,chrony,mariadb-client \
  --run-command "curl -fsSL https://get.docker.com | sh" \
  --run-command "systemctl enable ssh" \
  --run-command "systemctl enable docker" \
  --run-command "systemctl enable chrony" \
  --run-command "usermod -aG docker ubuntu" \
  --run-command "echo 'ubuntu ALL=(ALL) NOPASSWD:ALL' > /etc/sudoers.d/90-ubuntu-nopasswd" \
  --run-command "chmod 440 /etc/sudoers.d/90-ubuntu-nopasswd" \
  --run-command "python3 -m venv /opt/kolla-venv" \
  --run-command "/opt/kolla-venv/bin/pip install -U pip wheel" \
  --run-command "/opt/kolla-venv/bin/pip install 'ansible-core>=2.17,<2.18'" \
  --run-command "/opt/kolla-venv/bin/pip install 'kolla-ansible==2025.1.*'" \
  --run-command "/opt/kolla-venv/bin/kolla-ansible install-deps" \
  --timezone "Asia/Ho_Chi_Minh"

# Ghi chú:
# Nếu muốn dùng OpenStack release khác, đổi version kolla-ansible cho phù hợp.
# Ví dụ với 2025.2 thì nên kiểm tra lại yêu cầu ansible-core tương ứng.


### Verify Bước 14

Dùng `virt-cat` và `virt-ls` để kiểm tra nhanh nội dung bên trong image.


In [ ]:
# Kiểm tra Docker service file có trong image không
sudo virt-ls -a /var/lib/libvirt/templates/tpl-controller.img /etc/systemd/system/multi-user.target.wants/ | grep docker || true

# Kiểm tra SSH service
sudo virt-ls -a /var/lib/libvirt/templates/tpl-controller.img /etc/systemd/system/ | grep ssh || true

# Kiểm tra thư mục venv Kolla
sudo virt-ls -a /var/lib/libvirt/templates/tpl-controller.img /opt/ | grep kolla-venv

# Kiểm tra sudoers ubuntu
sudo virt-cat -a /var/lib/libvirt/templates/tpl-controller.img /etc/sudoers.d/90-ubuntu-nopasswd

# Nếu virt-customize fail:
# 1. Kiểm tra host có Internet không
# 2. Kiểm tra DNS
# 3. Kiểm tra libguestfs-tools đã cài chưa
# 4. Chạy lại với --verbose nếu cần debug


## BƯỚC 15 — Cấu hình SSH, mật khẩu và inject SSH key

Bước này cấu hình để sau khi clone VM từ template:

- Có thể SSH bằng user `ubuntu`
- Có thể đăng nhập bằng password
- Có thể SSH bằng key giảng viên/lab
- Root password được đặt để emergency access nếu cần

## Thông tin mặc định trong lab

- User: `ubuntu`
- Password: `123.abc`
- Root password: `123.abc`
- SSH key: `~/.ssh/lab_instructor_key`

## Cảnh báo production

Trong môi trường production thật:
- Không nên bật password SSH.
- Không nên đặt root password đơn giản.
- Nên dùng SSH key + disable password login.


In [ ]:
# ============================================================
# BƯỚC 15.1 — Tạo SSH key giảng viên/lab nếu chưa có
# ============================================================

# Tạo SSH key ed25519 dùng cho lab
# Nếu file đã tồn tại, lệnh này có thể hỏi overwrite.
# Không overwrite nếu key đang được dùng.
ssh-keygen -t ed25519 -f ~/.ssh/lab_instructor_key \
  -C "instructor@lab" -N ""

# Kiểm tra public key
cat ~/.ssh/lab_instructor_key.pub


In [ ]:
# ============================================================
# BƯỚC 15.2 — Bật SSH password login + set password + inject key
# ============================================================

cd /var/lib/libvirt/templates

sudo virt-customize -a tpl-controller.img \
  --run-command 'sed -i "s/^#PasswordAuthentication.*/PasswordAuthentication yes/" /etc/ssh/sshd_config' \
  --run-command 'sed -i "s/^PasswordAuthentication.*/PasswordAuthentication yes/" /etc/ssh/sshd_config' \
  --run-command 'sed -i "s/^#KbdInteractiveAuthentication.*/KbdInteractiveAuthentication yes/" /etc/ssh/sshd_config' \
  --run-command 'sed -i "s/^KbdInteractiveAuthentication.*/KbdInteractiveAuthentication yes/" /etc/ssh/sshd_config' \
  --run-command 'systemctl enable ssh' \
  --password "ubuntu:password:123.abc" \
  --root-password "password:123.abc" \
  --ssh-inject "ubuntu:file:$HOME/.ssh/lab_instructor_key.pub"

# Ghi chú:
# --password "ubuntu:password:123.abc" đặt password cho user ubuntu.
# --root-password đặt password root.
# --ssh-inject inject public key vào authorized_keys của user ubuntu.


### Verify Bước 15

Kiểm tra cấu hình SSH và authorized key trong image.


In [ ]:
# Kiểm tra PasswordAuthentication
sudo virt-cat -a /var/lib/libvirt/templates/tpl-controller.img /etc/ssh/sshd_config | grep -E "PasswordAuthentication|KbdInteractiveAuthentication"

# Kiểm tra authorized_keys của ubuntu
sudo virt-cat -a /var/lib/libvirt/templates/tpl-controller.img /home/ubuntu/.ssh/authorized_keys || true

# Kiểm tra user ubuntu tồn tại
sudo virt-cat -a /var/lib/libvirt/templates/tpl-controller.img /etc/passwd | grep ubuntu


## BƯỚC 16 — `virt-sysprep`: làm sạch template

`virt-sysprep` dùng để reset các thông tin định danh trước khi clone VM.

Mục tiêu:
- Xóa machine-id
- Xóa log/cached state không cần thiết
- Giữ lại SSH host keys nếu muốn, hoặc xóa để VM tự sinh lại

Trong lệnh này dùng:

```bash
--operations defaults,-ssh-hostkeys
```

Ý nghĩa:
- Chạy các operation mặc định.
- Không xóa SSH host keys.

Nếu muốn mỗi VM clone có SSH host key riêng, có thể bỏ `-ssh-hostkeys`, nhưng cần đảm bảo cloud-init/openssh sinh lại key đúng.


In [ ]:
# ============================================================
# BƯỚC 16 — Sysprep và bảo vệ template
# ============================================================

cd /var/lib/libvirt/templates

# Làm sạch template
sudo virt-sysprep -a tpl-controller.img \
  --operations defaults,-ssh-hostkeys

# Kiểm tra machine-id
# Kết quả mong muốn: trống hoặc "uninitialized"
sudo virt-cat -a tpl-controller.img /etc/machine-id || true

# Bảo vệ template: chỉ đọc, tránh sửa nhầm
sudo chmod 444 tpl-controller.img

# Kiểm tra quyền file
ls -lh tpl-controller.img


### Verify Bước 16

Xác nhận template đã sẵn sàng để dùng làm base clone.


In [ ]:
# Verify quyền read-only
ls -lh /var/lib/libvirt/templates/tpl-controller.img

# Verify image format và size
qemu-img info /var/lib/libvirt/templates/tpl-controller.img

# Verify machine-id
sudo virt-cat -a /var/lib/libvirt/templates/tpl-controller.img /etc/machine-id || true

# Kiểm tra nhanh các thành phần chính
sudo virt-ls -a /var/lib/libvirt/templates/tpl-controller.img /opt/ | grep kolla-venv
sudo virt-cat -a /var/lib/libvirt/templates/tpl-controller.img /etc/sudoers.d/90-ubuntu-nopasswd


## Checklist hoàn thành PHẦN III

Đánh dấu trước khi chuyển sang phần tạo Controller VM thật:

- [ ] `ubuntu-24.04-base.img` đã tải thành công.
- [ ] Checksum image OK.
- [ ] `tpl-controller.img` đã resize lên 100GB.
- [ ] `virt-customize` chạy xong không lỗi.
- [ ] Docker đã được enable trong image.
- [ ] SSH đã được enable trong image.
- [ ] User `ubuntu` có sudo NOPASSWD.
- [ ] `/opt/kolla-venv` tồn tại.
- [ ] Kolla-Ansible đã được cài trong venv.
- [ ] SSH key lab đã được inject.
- [ ] Password login đã bật cho lab.
- [ ] `virt-sysprep` đã chạy.
- [ ] Template đã được chmod `444`.

## Lỗi hay gặp

### 1. `virt-customize: command not found`

Cài thiếu package:

```bash
sudo apt install -y libguestfs-tools
```

### 2. `virt-customize` treo lâu

Thường do:
- DNS lỗi
- Host không ra Internet
- apt mirror chậm
- pip không tải được package

Kiểm tra:

```bash
ping -c 3 8.8.8.8
ping -c 3 archive.ubuntu.com
curl -I https://pypi.org
```

### 3. SSH vào VM clone không được

Kiểm tra:
- Cloud-init có chạy không
- User `ubuntu` tồn tại không
- Security/firewall
- VM có IP chưa
- PasswordAuthentication đã bật chưa

### 4. Clone từ template nhưng sửa luôn template

Để tránh lỗi này, template đã được đặt read-only:

```bash
sudo chmod 444 tpl-controller.img
```

Khi tạo VM thật, nên copy hoặc dùng backing file, không boot trực tiếp template.
